# Enam eksperimen utama: siap, belum dieksekusi

Training menunggu otorisasi pengguna; baseline menunggu persetujuan tim. Tidak menjalankan subgroup/reweighting. Lihat RUNNER_CHANGES.md.

In [1]:
import os
from pathlib import Path
assert os.environ.get("TTKI_P1_TRAINING_AUTHORIZED")=="reproduction-001-six-main", "Training belum diotorisasi pengguna"
assert Path.cwd().resolve()==Path('[REPOSITORY_ROOT]\\runs\\reproduction-001').resolve()
exec(compile(Path("stage5_preflight.py").read_text(encoding="utf-8"),"stage5_preflight.py","exec"),{"__name__":"__main__","__file__":str(Path("stage5_preflight.py").resolve())})


{
  "status": "PASS",
  "training_started": false,
  "approval": "pending team; user training authorization required",
  "shape": [
    235370,
    42
  ],
  "cache_sha256": "e7678de14cae2ed296d22d2b2644196af551ae12550dab5170413ce000ea3d8a",
  "membership_sha256": "9fa7056edd95982ac00adc3bb4e70cbeac6948296c6998cea318d6ca0fe8c8f6",
  "lock_sha256": "b49bafc8e76d16feb0e8b3102873aa66b0ceb779eeda2177d2c8304a9276ceea",
  "python": "3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]",
  "packages": {
    "numpy": "2.5.2",
    "pandas": "3.0.5",
    "scikit-learn": "1.9.1",
    "matplotlib": "3.11.1",
    "tldextract": "5.3.1",
    "threadpoolctl": "3.8.0",
    "ipykernel": "7.4.0"
  },
  "feature_counts": [
    38,
    15,
    15
  ],
  "C_domain_overlap": 0
}


In [2]:
from importlib.metadata import version
from pathlib import Path
from statistics import median
from time import perf_counter
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

INPUT=Path("hasil")
OUT=Path("eksperimen_utama")
assert not OUT.exists(), "Output exists; do not overwrite"
OUT.mkdir()
(OUT/"predictions").mkdir()
manifest = json.loads((INPUT / "manifest_data.json").read_text(encoding="utf-8"))
data = pd.read_csv(INPUT / "fitur_internal.csv.gz")
SEED = manifest["seed"]
LEXICAL = manifest["fitur_leksikal"]
PAGE = manifest["fitur_halaman"]
GROUPS = ("akar", "path", "query")
FPR_LIMIT = .01
MIN_SUPPORT = 30
BATCH = 1024
PARAMS = {
    "LR": dict(max_iter=1000, random_state=SEED),
    "RF": dict(n_estimators=60, max_depth=16, min_samples_leaf=3, random_state=SEED, n_jobs=1),
}
import hashlib, traceback, warnings
from datetime import datetime
from zoneinfo import ZoneInfo
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
members=pd.read_csv("split_membership.csv.gz")
main_rows=[]
actual_parameters={}
config={"A":("acak",LEXICAL+PAGE),"B":("acak",LEXICAL),"C":("domain",LEXICAL)}
assert config["A"][0]==config["B"][0] and config["B"][1]==config["C"][1]


Matplotlib is building the font cache; this may take a moment.


In [3]:
def choose_threshold(y, scores, fpr_limit):
    order = np.argsort(-scores, kind="stable")
    score, target = scores[order], y[order]
    ends = np.r_[np.flatnonzero(score[:-1] != score[1:]), len(y)-1]
    thresholds = np.r_[np.nextafter(score[0], np.inf), score[ends]]
    tp = np.r_[0, np.cumsum(target)[ends]]
    fp = np.r_[0, ends+1] - tp
    positives = y.sum()
    fpr = fp / (len(y)-positives)
    if fpr_limit is None:
        goal, allowed = 2*tp/(positives+tp+fp), np.arange(len(tp))
    else:
        goal, allowed = tp/positives, np.flatnonzero(fpr <= fpr_limit)
    best = max(allowed, key=lambda i: (goal[i], -fpr[i], thresholds[i]))
    return float(thresholds[best])

def metrics(y, scores, threshold):
    tn, fp, fn, tp = confusion_matrix(y, scores >= threshold, labels=[0, 1]).ravel()
    pos, neg = tp+fn, tn+fp
    return dict(tn=tn, fp=fp, fn=fn, tp=tp, phishing=pos, sah=neg,
                precision=tp/(tp+fp) if tp+fp else 0.0,
                recall=tp/pos if pos else np.nan, fpr=fp/neg if neg else np.nan,
                f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0,
                roc_auc=roc_auc_score(y, scores) if pos and neg else np.nan)

def fit_and_score(name, split, columns, strategy):
    assert strategy == "tanpa_bobot"
    train, val, test = (data.loc[data[split] == part] for part in ("latih", "validasi", "uji"))
    if name == "LR":
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(**PARAMS[name]))
    else:
        model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(**PARAMS[name]))
    fit_args = {}

    with threadpool_limits(limits=1):
        training_start = perf_counter()
        model.fit(train[columns], train["phishing"], **fit_args)
        training_seconds = perf_counter()-training_start
        assert np.array_equal(model.classes_, [0,1])
        batch = test[columns].sample(n=BATCH, random_state=SEED)
        model.predict_proba(batch)
        times = []
        for _ in range(5):
            start = perf_counter()
            model.predict_proba(batch)
            times.append((perf_counter()-start)*1000)
        val_scores = model.predict_proba(val[columns])[:, 1]
        test_scores = model.predict_proba(test[columns])[:, 1]
    return val, test, val_scores, test_scores, median(times), model, training_seconds

In [4]:
for name in ("LR","RF"):
    for scenario,(split,columns) in config.items():
        started=datetime.now(ZoneInfo("Asia/Bangkok")).isoformat()
        try:
            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter("always")
                val,test,vs,ts,elapsed,model,train_seconds=fit_and_score(name,split,columns,"tanpa_bobot")
            threshold=choose_threshold(val["phishing"].to_numpy(),vs,None)
            row=dict(model=name,skenario=scenario,ambang=threshold,training_seconds=train_seconds,uji_batch_ms=elapsed,batch_size=BATCH)
            for prefix,subset,scores in [("val",val,vs),("uji",test,ts)]:
                y=subset["phishing"].to_numpy()
                values=metrics(y,scores,threshold)
                values.update(samples=len(subset),accuracy=float((y==(scores>=threshold)).mean()))
                row.update({f"{prefix}_{key}":value for key,value in values.items()})
                pred=members.loc[subset.index,["clean_row_id","url_sha256","phishing"]].copy()
                pred["probability_phishing"]=scores
                pred["threshold"]=threshold
                pred["prediction"]=(scores>=threshold).astype(int)
                pred.to_csv(OUT/"predictions"/f"{name}_{scenario}_{prefix}.csv.gz",index=False)
            actual_parameters[f"{name}/{scenario}"]={"pipeline":{step:est.get_params(deep=False) for step,est in model.steps},"classes":model.classes_.tolist(),"features":columns,"split":split,"warnings":[str(w.message) for w in captured]}
            main_rows.append(row)
            pd.DataFrame(main_rows).to_csv(OUT/"metrik_utama.csv",index=False)
            (OUT/"actual_parameters.json").write_text(json.dumps(actual_parameters,indent=2,default=str),encoding="utf-8")
            event=dict(model=name,scenario=scenario,started=started,completed=datetime.now(ZoneInfo("Asia/Bangkok")).isoformat(),training_seconds=train_seconds,status="PASS",warnings=[str(w.message) for w in captured])
            with (OUT/"training_log.jsonl").open("a",encoding="utf-8") as log:log.write(json.dumps(event)+chr(10))
        except Exception:
            with (OUT/"training_log.jsonl").open("a",encoding="utf-8") as log:log.write(json.dumps(dict(model=name,scenario=scenario,status="ERROR",traceback=traceback.format_exc()))+chr(10))
            raise
assert len(main_rows)==6
main=pd.DataFrame(main_rows)
deltas=[]
for name,subset in main.groupby("model"):
    values=subset.set_index("skenario")
    for later,earlier in [("B","A"),("C","B")]:
        deltas.append(dict(model=name,perbandingan=f"{later} - {earlier}",**{col:values.loc[later,col]-values.loc[earlier,col] for col in ["uji_f1","uji_recall","uji_fpr"]}))
pd.DataFrame(deltas).to_csv(OUT/"selisih_utama.csv",index=False)


In [5]:
checks=[]
for row in main_rows:
    for prefix in ["val","uji"]:
        p=pd.read_csv(OUT/"predictions"/f"{row['model']}_{row['skenario']}_{prefix}.csv.gz")
        y=p.phishing.to_numpy();score=p.probability_phishing.to_numpy();pred=(score>=row["ambang"]).astype(int)
        assert len(p)==row[prefix+"_samples"] and p.url_sha256.is_unique and np.array_equal(pred,p.prediction)
        tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
        independent=dict(tn=tn,fp=fp,fn=fn,tp=tp,accuracy=accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred),f1=f1_score(y,pred),fpr=fp/(fp+tn),roc_auc=roc_auc_score(y,score))
        for metric,value in independent.items():
            assert np.isclose(value,row[prefix+"_"+metric],rtol=0,atol=1e-12)
            checks.append(dict(model=row["model"],scenario=row["skenario"],part=prefix,metric=metric,status="PASS"))
pd.DataFrame(checks).to_csv(OUT/"independent_metric_validation.csv",index=False)
historical=pd.read_csv(Path('[REPOSITORY_ROOT]')/"hasil/metrik_utama.csv")
comparisons=[]
for row in main_rows:
    old=historical.loc[(historical.model==row["model"])&(historical.skenario==row["skenario"])].iloc[0]
    for metric in ["ambang","val_f1","val_precision","val_recall","val_fpr","val_roc_auc","uji_f1","uji_precision","uji_recall","uji_fpr","uji_roc_auc","uji_tn","uji_fp","uji_fn","uji_tp","uji_batch_ms"]:
        comparisons.append(dict(model=row["model"],scenario=row["skenario"],metric=metric,historical=old[metric],reproduction=row[metric],absolute_difference=abs(row[metric]-old[metric]),note="Hardware dependent" if metric=="uji_batch_ms" else "Historical per-record predictions unavailable"))
pd.DataFrame(comparisons).to_csv(OUT/"historical_comparison.csv",index=False)
metadata=dict(status="completed",baseline_approval="pending team",run_id="reproduction-001",seed=SEED,params=PARAMS,inputs=json.loads(Path("stage5_preflight.json").read_text()),python=platform.python_version(),cpu=platform.processor(),os=platform.platform(),timing="batch1024; warmup1; median5; threads1; pipeline preprocessing+predict_proba; excludes URL extraction and page retrieval",models_saved=False,model_storage_reason="Optional; full parameters and predictions retained",membership_hashes={})
for scenario,(split,columns) in config.items():
    for part in ["latih","validasi","uji"]:
        ids=members.loc[members[split]==part,"url_sha256"].tolist()
        metadata["membership_hashes"][f"{scenario}/{part}"]=hashlib.sha256(chr(10).join(ids).encode()).hexdigest()
(OUT/"manifest_eksperimen.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
checksums={p.relative_to(OUT).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob("*") if p.is_file()}
(OUT/"artifact_checksums.json").write_text(json.dumps(checksums,indent=2),encoding="utf-8")
display(main)


,model,skenario,ambang,training_seconds,uji_batch_ms,batch_size,val_tn,val_fp,val_fn,val_tp,...,uji_tp,uji_phishing,uji_sah,uji_precision,uji_recall,uji_fpr,uji_f1,uji_roc_auc,uji_samples,uji_accuracy
0,LR,A,0.415657,2.396193,4.1047,1024,20216,11,9,15069,...,15066,15078,20228,0.999337,0.999204,0.000494,0.999270,0.999975,35306,0.999377
1,LR,B,0.506426,0.850477,2.7464,1024,20220,7,245,14833,...,14874,15078,20228,0.999530,0.986470,0.000346,0.992957,0.996919,35306,0.994024
2,LR,C,0.574050,1.108267,2.5335,1024,20286,3,227,13840,...,15484,15744,20485,0.999355,0.983486,0.000488,0.991357,0.996278,36229,0.992547
3,RF,A,0.510750,17.285501,10.8439,1024,20227,0,1,15077,...,15072,15078,20228,1.000000,0.999602,0.000000,0.999801,0.999998,35306,0.999830
4,RF,B,0.559311,6.000497,9.9760,1024,20220,7,158,14920,...,14951,15078,20228,0.998998,0.991577,0.000742,0.995274,0.998093,35306,0.995978
5,RF,C,0.555678,5.862491,9.4296,1024,20279,10,146,13921,...,15575,15744,20485,0.999358,0.989266,0.000488,0.994286,0.997540,36229,0.995059
